# Fine-tuning Granite LLM for Employee Security Analysis

This notebook fine-tunes IBM Granite model for cybersecurity behavior analysis using LoRA/QLoRA.

## 1. Install Dependencies

In [ ]:
!pip install --upgrade pip
!pip install torch==2.1.0
!pip install transformers==4.36.0
!pip install datasets==2.16.0
!pip install peft==0.7.0
!pip install accelerate==0.25.0
!pip install bitsandbytes==0.41.3
!pip install trl==0.7.4
!pip install scipy

## 2. Import Libraries

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline
)
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer
import json

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 3. Configuration

In [ ]:
MODEL_NAME = "ibm-granite/granite-3b-code-instruct"
OUTPUT_DIR = "../model/granite-security-finetuned"
DATASET_PATH = "../datasets/security_training_data.jsonl"

LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
MAX_SEQ_LENGTH = 2048

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

print("Configuration loaded successfully")

## 4. Load Training Dataset

In [ ]:
print("Loading training dataset...")

dataset = load_dataset('json', data_files=DATASET_PATH, split='train')

print(f"Total examples: {len(dataset)}")
print("\nFirst training example:")
print(json.dumps(dataset[0], indent=2))

## 5. Format Dataset for Training

In [ ]:
def format_chat_template(example):
    messages = example['messages']
    formatted_text = f"""<|system|>
{messages[0]['content']}

<|user|>
{messages[1]['content']}

<|assistant|>
{messages[2]['content']}"""
    return {"text": formatted_text}

dataset = dataset.map(format_chat_template, remove_columns=dataset.column_names)
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset['train']
eval_dataset = dataset['test']

print(f"Training examples: {len(train_dataset)}")
print(f"Validation examples: {len(eval_dataset)}")
print("\nFormatted example:")
print(train_dataset[0]['text'][:500] + "...")

## 6. Load Base Model with Quantization

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("\nConfiguring quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("\nLoading base model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)
print("✓ Model loaded")

## 7. Configure LoRA

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 8. Setup Training

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    optim="paged_adamw_32bit",
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=10,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    fp16=True,
    push_to_hub=False,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    tokenizer=tokenizer,
    args=training_args,
    packing=False,
)

print("✓ Trainer initialized")

## 9. Start Training

In [ ]:
import time
start_time = time.time()

trainer.train()

training_time = time.time() - start_time
print(f"\nTraining time: {training_time/60:.2f} minutes")

## 10. Save Model

In [ ]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✓ Model saved to: {OUTPUT_DIR}")

## 11. Test Model

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
model = model.merge_and_unload()

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.7,
    top_p=0.9,
    do_sample=True
)

print("✓ Model loaded for testing")

In [ ]:
test_prompt = """<|system|>
You are a cybersecurity expert.

<|user|>
Analyze: Employee emp_TEST had 4 after-hours accesses from external IP 85.100.200.50, downloaded customer_database.sql

<|assistant|>
"""

result = pipe(test_prompt)
response = result[0]['generated_text'].split('<|assistant|>')[-1].strip()
print(response)

## 12. Export for vLLM

In [ ]:
merged_model_path = OUTPUT_DIR + "-merged"
model.save_pretrained(merged_model_path)
tokenizer.save_pretrained(merged_model_path)
print(f"✓ Merged model saved: {merged_model_path}")
print("\nReady for vLLM deployment!")